In [6]:
import requests
from pymilvus import MilvusClient

In [105]:
from ollama import Client

In [120]:
client = MilvusClient(uri="http://localhost:19530")

In [3]:
# client.drop_collection(collection_name='questions')

In [194]:
client.drop_collection(collection_name='tables')

In [193]:
client.drop_collection(collection_name='datas')

In [7]:
headers = {
    'Content-Type': 'application/json',
}

In [8]:
def get_vector(question):
    json_data = {'inputs': question}
    r = requests.post('http://localhost:8081/embed', headers=headers, json=json_data)
    que = r.json()[0]
    return que

In [9]:
que = get_vector('洗衣机')

In [99]:
# que = get_vector('For all the users who gave "A Shot in the Dark" a rating, how many percent of them is a paying subscriber?')
# que = get_vector('For all the users who gave "A Shot in the Dark" a rating, and paying subscribers.')
# que = get_vector("What percentage of 'A Shot in the Dark' raters are paying subscribers?")
que = get_vector("movies")
# que = get_vector('A Shot in the Dark')
# 记录过故障日志但没有联系任何来访工程师的工作人员的姓名和id是什么？ database_5_assets_maintenance
# search_params={"metric_type": "IP"},

In [121]:
res = client.search(
    collection_name="tables",
    data=[que],
    limit=3,
    partition_names = ['database_2_movie_platform'],
    output_fields=['text']
)

In [103]:
res = client.search(
    collection_name="tables",
    data=[que],
    anns_field="vector",
    limit=3,
    partition_names = ['database_2_movie_platform'],
    output_fields=['text'],
    filter = 'table=="movies"'
)

In [132]:
res = client.search(
    collection_name="datas",
    data=[que],
    anns_field="vector",
    limit=10,
    partition_names = ['database_2_movie_platform'],
    output_fields=['text', 'table'],
    # filter='table == "movies"',
    # search_params={"metric_type": "IP"}
)

In [ ]:
res = client.search(
    collection_name="questions",
    data=[que],
    anns_field="vector",
    limit=5,
    output_fields=['text'],
    filter='db == "database_2_movie_platform"',
)

In [133]:
for hits in res:
    print("TopK results:")
    for hit in hits:
        print('%s\n' % hit)

TopK results:
{'id': 3522, 'distance': 0.4857076108455658, 'entity': {'text': '{"movie_id": 6419, "movie_title": "A Shot in the Dark", "movie_release_year": 1964, "movie_url": "http://mubi.com/films/a-shot-in-the-dark", "movie_title_language": "en", "movie_popularity": 90, "movie_image_url": "https://images.mubicdn.net/images/film/6419/cache-26330-1445883321/image-w1280.jpg", "director_id": "5027", "director_name": "Blake Edwards", "director_url": "http://mubi.com/cast/blake-edwards"}', 'table': 'movies'}}

{'id': 3523, 'distance': 0.4842716157436371, 'entity': {'text': '{"movie_id": 133720, "movie_title": "A Shot in the Dark", "movie_release_year": 2005, "movie_url": "http://mubi.com/films/a-shot-in-the-dark-2005", "movie_title_language": "en", "movie_popularity": 0, "movie_image_url": "https://images.mubicdn.net/images/film/133720/cache-119646-1445971029/image-w1280.jpg", "director_id": "61552", "director_name": "Leonel Vieira", "director_url": "http://mubi.com/cast/leonel-vieira"}',

In [116]:
ollama_client = Client(
  host='http://localhost:11434',
  headers={'Content-Type': 'application/json',}
)

In [155]:
def get_chat_ollama_body(prompt):
    response = ollama_client.chat(
        model='qwen2.5-coder:14b', 
        messages=[  # qwen2.5-coder:7b
            {'role': 'user', 'content': prompt},],
        options={"temperature":0.1}
    )

    content = response.message.content
    return content

In [110]:
get_chat_ollama_body('精简以下句子，保留核心意思：For all the users who gave "A Shot in the Dark" a rating, how many percent of them is a paying subscriber?')

'What percentage of users who rated "A Shot in the Dark" are paying subscribers?'

In [179]:
que_text = get_chat_ollama_body('抽取这句话的核心词汇(只列出词，不要其他描述)：For all the users who gave "A Shot in the Dark" a rating, how many percent of them is a paying subscriber?')
que = get_vector(que_text)
print(que_text)

users, "A Shot in the Dark", rating, paying subscriber


In [180]:
res = client.search(
    collection_name="datas",
    data=[que],
    anns_field="vector",
    limit=10,
    partition_names = ['database_2_movie_platform'],
    output_fields=['table'],
    # filter='table == "movies"',
    # search_params={"metric_type": "IP"}
)

for hits in res:
    print("TopK results:")
    for hit in hits:
        print('%s\n' % hit)

TopK results:
{'id': 3523, 'distance': 0.5290443301200867, 'entity': {'table': 'movies'}}

{'id': 3524, 'distance': 0.524229109287262, 'entity': {'table': 'movies'}}

{'id': 3522, 'distance': 0.5194863677024841, 'entity': {'table': 'movies'}}

{'id': 3541, 'distance': 0.5085480213165283, 'entity': {'table': 'ratings_users'}}

{'id': 3542, 'distance': 0.5077122449874878, 'entity': {'table': 'ratings_users'}}

{'id': 3534, 'distance': 0.5062206387519836, 'entity': {'table': 'ratings_users'}}

{'id': 3535, 'distance': 0.5045004487037659, 'entity': {'table': 'ratings_users'}}

{'id': 3544, 'distance': 0.5041618943214417, 'entity': {'table': 'ratings_users'}}

{'id': 3543, 'distance': 0.5035110116004944, 'entity': {'table': 'ratings_users'}}

{'id': 3536, 'distance': 0.5020223259925842, 'entity': {'table': 'ratings_users'}}



In [183]:
res = client.search(
    collection_name="tables",
    data=[que],
    limit=3,
    partition_names = ['database_2_movie_platform'],
    output_fields=['text', 'table'],
)

for hits in res:
    print("TopK results:")
    for hit in hits:
        print('%s\n' % hit['entity'])

TopK results:
{'text': 'create table ratings_users ("user_id" INTEGER, "rating_date_utc" TEXT, "user_trialist" INTEGER, "user_subscriber" INTEGER, "user_avatar_image_url" TEXT, "user_cover_image_url" TEXT, "user_eligible_for_trial" INTEGER, "user_has_payment_method" INTEGER, FOREIGN KEY ("user_id") REFERENCES "lists_users" ("user_id") ON DELETE NO ACTION ON UPDATE NO ACTION);', 'table': 'ratings_users'}

{'text': 'create table users ("id" INT, "is_paying_subscriber" BOOLEAN, PRIMARY KEY ("id"));', 'table': 'users'}

{'text': 'create table ratings ("movie_id" INTEGER, "rating_id" INTEGER, "rating_url" TEXT, "rating_score" INTEGER, "rating_timestamp_utc" TEXT, "critic" TEXT, "critic_likes" INTEGER, "critic_comments" INTEGER, "user_id" INTEGER, "user_trialist" INTEGER, "user_subscriber" INTEGER, "user_eligible_for_trial" INTEGER, "user_has_payment_method" INTEGER, FOREIGN KEY ("user_id") REFERENCES "ratings_users" ("user_id") ON DELETE NO ACTION ON UPDATE NO ACTION, FOREIGN KEY ("rating_i

In [185]:
res = client.search(
        collection_name="questions",
        data=[que],
        anns_field="vector",
        limit=4,
        # partition_names = [partition_name],
        output_fields=['text'],
        filter='db=="database_2_movie_platform"'
    )

for hits in res:
    print("TopK results:")
    for hit in hits:
        print('%s\n' % hit)

TopK results:
{'id': 2062, 'distance': 0.5035185813903809, 'entity': {'text': "What's the number of the paying subscribers when rating a movie after the year 2014?\nSELECT COUNT(user_subscriber) FROM ratings_users WHERE user_has_payment_method = 1 AND rating_date_utc > '2014%'"}}

{'id': 1966, 'distance': 0.49864622950553894, 'entity': {'text': 'For movie id 1269, how many users, who was a paying subscriber and was eligible for trial when he rated the movie, gave the movie a rating score of less than or equal to 2?\nSELECT COUNT(*) FROM ratings WHERE movie_id = 1269 AND rating_score <= 2 AND user_eligible_for_trial = 1 AND user_has_payment_method = 1'}}

{'id': 1954, 'distance': 0.4898223876953125, 'entity': {'text': 'What is the percentage of the ratings were rated by user who was a subcriber?\nSELECT CAST(SUM(CASE WHEN user_subscriber = 1 THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(*) FROM ratings'}}

{'id': 2048, 'distance': 0.48268210887908936, 'entity': {'text': 'How many users were

In [186]:
que = get_vector('create table movies')
res = client.search(
    collection_name="tables",
    data=[que],
    limit=3,
    partition_names = ['database_2_movie_platform'],
    output_fields=['table'], # 'text', 
    filter='table == "movies"'
)

for hits in res:
    print("TopK results:")
    for hit in hits:
        print('%s\n' % hit)

TopK results:
{'id': 268, 'distance': 0.7765492796897888, 'entity': {'table': 'movies'}}



### aaa

In [6]:
import requests
from llama_index.llms.base import LLM
from llama_index.llms.types import ChatMessage, ChatResponse, MessageRole

class OllamaLLM(LLM):
    def __init__(self, api_url: str, model_name: str):
        self.api_url = api_url
        self.model_name = model_name

    def chat(self, messages: list[ChatMessage], **kwargs) -> ChatResponse:
        # 将消息转换为 Ollama API 支持的格式
        payload = {
            "model": self.model_name,
            "messages": [{"role": msg.role, "content": msg.content} for msg in messages],
        }
        response = requests.post(f"{self.api_url}/chat", json=payload)
        response.raise_for_status()
        data = response.json()
        return ChatResponse(
            messages=[
                ChatMessage(role=MessageRole.ASSISTANT, content=data["response"])
            ]
        )

ModuleNotFoundError: No module named 'llama_index.llms.base'

In [5]:
from llama_index.core import SQLDatabase
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    String,
    Integer,
    select,
)
from llama_index.core.query_engine import NLSQLTableQueryEngine

In [3]:
engine = create_engine("sqlite:///F:/2024技能大赛/databases/database_1/中国城市.sqlite")

In [4]:
sql_database = SQLDatabase(engine, include_tables=["城市", "省份", "省生产总值", "市生产总值"])

In [ ]:
query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database, tables=["城市", "省份", "省生产总值", "市生产总值"], llm=llm
)
query_str = "哪个城市的人口最多？"
response = query_engine.query(query_str)